💡 **Environment:** `clamp-analyses`  


# Description

Predicts drug-disease associations using the GTEx **module-based** CLAMP model.

Reads LINCS and S-PrediXcan projections from:
- `02_spredixcan_projection_gtex` (S-PrediXcan, LVs × traits)
- `03_lincs_projection_gtex` (LINCS, LVs × drugs)

For each of 49 tissues and 5 LV-count thresholds (all, 5, 10, 25, 50), scores:
$$\text{score} = -1 \times \mathbf{drug}^T \mathbf{disease}$$

# Modules loading


In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd

from pyprojroot import here

# Settings


In [ ]:
PREDICTION_METHOD = 'module_based_gtex'
MODEL_KEY = PREDICTION_METHOD.removeprefix('module_based_')

In [ ]:
DATA_DIR = here('data/drug_disease_associations')
display(DATA_DIR)
assert DATA_DIR.exists()

NB_NAME = '08_prediction_module_based_gtex'
OUTPUT_DIR = here('output/03_model_biology/00_archs4/02_drug_disease_associations/' + NB_NAME)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Inputs from upstream projection notebooks
LINCS_PROJ_FILE = here('output/03_model_biology/00_archs4/02_drug_disease_associations/03_lincs_projection_gtex') / 'lincs' / 'lincs-projection.pkl'
display(LINCS_PROJ_FILE)
assert LINCS_PROJ_FILE.exists()

SPREDIXCAN_PROJ_DIR = here('output/03_model_biology/00_archs4/02_drug_disease_associations/02_spredixcan_projection_gtex') / 'spredixcan'
display(SPREDIXCAN_PROJ_DIR)
assert SPREDIXCAN_PROJ_DIR.exists()

OUTPUT_PREDICTIONS_DIR = OUTPUT_DIR / 'lincs' / 'predictions'
OUTPUT_PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
display(OUTPUT_PREDICTIONS_DIR)

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/data/drug_disease_associations')

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/03_lincs_projection_gtex/lincs/lincs-projection.pkl')

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/02_spredixcan_projection_gtex/spredixcan')

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/08_prediction_module_based_gtex/lincs/predictions')

# Helper functions


In [5]:
import sys
sys.path.insert(0, str(here('libs')))
from drug_disease_utils import map_traits_to_doid, _zero_nontop_genes, predict_dotprod_neg

# Load PharmacotherapyDB gold standard


In [6]:
gold_standard = pd.read_pickle(DATA_DIR / 'gold_standard.pkl')
display(gold_standard.shape)
display(gold_standard.head())

doids_in_gold_standard = set(gold_standard['trait'])
print(f'Unique DOIDs in gold standard: {len(doids_in_gold_standard)}')

(998, 3)

,trait,drug,true_class
0,DOID:10652,DB00843,1
1,DOID:10652,DB00674,1
2,DOID:10652,DB01043,1
3,DOID:10652,DB00989,1
4,DOID:10652,DB00810,0


Unique DOIDs in gold standard: 87


# Load trait → DOID mapping files


In [ ]:
ukb_efo = pd.read_csv(
    DATA_DIR / 'phenomexcan_traits_fullcode_to_efo.tsv',
    sep='\t',
    index_col='ukb_fullcode',
)
# PhenoPlier stores trait full codes with hyphens (e.g. "I70-Diagnoses_...") but
# our S-PrediXcan data uses underscores throughout (e.g. "I70_Diagnoses_...").
# Normalize the index
ukb_efo.index = [idx.replace('-', '_', 1) for idx in ukb_efo.index]

efo_xrefs = pd.read_csv(DATA_DIR / 'term_id_xrefs.tsv.gz', sep='\t')
do_xrefs = pd.read_csv(DATA_DIR / 'xrefs-prop-slim.tsv', sep='\t')

# Load LINCS projection


In [ ]:
lincs_projection = pd.read_pickle(LINCS_PROJ_FILE)
print(f'LINCS projection shape: {lincs_projection.shape}')
assert not lincs_projection.isna().any().any()
display(lincs_projection.head())

LINCS projection shape: (578, 1170)


perturbagen,DB00014,DB00091,DB00121,DB00130,DB00131,DB00132,DB00136,DB00140,DB00146,DB00150,...,DB08995,DB09002,DB09004,DB09009,DB09010,DB09015,DB09019,DB09020,DB09022,DB09023
LV1,0.002925,-0.084773,0.013740,0.009346,0.000841,0.011196,-0.027155,-0.014251,-0.004858,0.003378,...,-0.008925,0.015471,-0.018164,0.009511,-0.002329,-0.011535,0.015061,-0.013588,-0.008417,0.008109
LV2,0.004306,-0.119468,0.116219,-0.004673,-0.022354,0.095246,0.031966,-0.009553,0.012612,-0.011840,...,0.013659,0.041424,0.048180,-0.024304,-0.055466,-0.006957,0.034995,-0.008117,0.036984,-0.022943
LV3,-0.053713,-0.199456,0.064356,-0.025178,0.015408,-0.003633,-0.026251,0.014058,-0.072818,-0.019195,...,-0.030883,0.050052,0.042901,-0.008591,-0.032787,-0.014382,-0.023704,0.024361,0.050356,-0.005251
LV4,-0.010796,0.272352,-0.020650,0.028555,-0.036441,-0.015145,-0.064205,0.031375,0.023564,-0.011486,...,-0.037756,-0.000161,-0.021790,-0.038158,0.048355,0.038748,-0.039573,-0.068299,-0.026051,0.007173
LV5,-0.020548,0.181593,-0.000023,0.068556,-0.056003,0.012284,0.106527,0.032506,0.070166,-0.001743,...,-0.044886,-0.012859,-0.024814,-0.026362,0.030894,0.010696,-0.021258,0.073416,0.022983,0.021485


# Load S-PrediXcan projected files


In [ ]:
spredixcan_file_list = sorted(
    f for f in SPREDIXCAN_PROJ_DIR.glob(f'spredixcan-*-projection-{MODEL_KEY}.pkl')
)
display(len(spredixcan_file_list))
assert len(spredixcan_file_list) == 49

display(pd.read_pickle(spredixcan_file_list[0]).head())

49

,100001_raw_Food_weight,100002_raw_Energy,100003_raw_Protein,100004_raw_Fat,100005_raw_Carbohydrate,100006_raw_Saturated_fat,100007_raw_Polyunsaturated_fat,100008_raw_Total_sugars,100009_raw_Englyst_dietary_fibre,100010_Portion_size,...,Z50_Diagnoses_main_ICD10_Z50_Care_involving_use_of_rehabilitation_procedures,Z51_Diagnoses_main_ICD10_Z51_Other_medical_care,Z52_Diagnoses_main_ICD10_Z52_Donors_of_organs_and_tissues,Z53_Diagnoses_main_ICD10_Z53_Persons_encountering_health_services_for_specifie_procedures_not_carried_out,Z71_Diagnoses_main_ICD10_Z71_Persons_encountering_health_services_for_other_counselling_and_medical_advice_not_elsewhere_classified,Z76_Diagnoses_main_ICD10_Z76_Persons_encountering_health_services_in_other_circumstances,Z80_Diagnoses_main_ICD10_Z80_Family_history_of_malignant_neoplasm,Z85_Diagnoses_main_ICD10_Z85_Personal_history_of_malignant_neoplasm,Z87_Diagnoses_main_ICD10_Z87_Personal_history_of_other_diseases_and_conditions,pgc_scz2
LV1,0.021172,0.016790,0.008607,0.012632,0.029574,0.020426,0.005847,0.025923,0.043708,0.016406,...,0.020754,-0.001633,0.001944,0.017229,-0.027172,-0.004984,0.021795,-0.009339,-0.020161,0.051531
LV2,0.054331,0.013277,0.011917,0.000119,0.020174,-0.012230,-0.016386,0.005693,0.061851,0.007690,...,0.010018,0.024360,0.041070,-0.015142,0.033289,0.011230,0.006908,-0.000141,-0.052771,0.063697
LV3,-0.027630,-0.042979,-0.014214,-0.056511,0.003154,-0.046968,-0.039777,0.003957,0.060750,0.008270,...,0.023798,0.030875,-0.073629,-0.108407,-0.033537,-0.037273,-0.013678,0.015349,-0.074574,0.013474
LV4,-0.030432,-0.071636,-0.049565,-0.016407,-0.084217,-0.018471,-0.032933,-0.056530,-0.014455,-0.051154,...,0.037046,-0.023586,-0.024877,-0.051551,-0.005466,-0.012831,0.013481,-0.039162,-0.064293,0.002347
LV5,0.050751,0.016303,0.024483,0.017029,0.001495,-0.005634,0.051685,0.001537,0.050363,0.013703,...,-0.034870,0.035624,0.011072,0.002469,-0.009455,-0.030543,-0.043036,0.026544,-0.013861,-0.018270


# Predict drug-disease associations


In [ ]:
N_TOP_LVS_LIST = [None, 5, 10, 25, 50]

for spredixcan_file in spredixcan_file_list:
    print(spredixcan_file.name)

    tissue_proj = pd.read_pickle(spredixcan_file)
    print(f'  shape: {tissue_proj.shape}')
    assert tissue_proj.index.equals(lincs_projection.index)

    for ntc in N_TOP_LVS_LIST:
        predict_dotprod_neg(
            lincs_projection,
            spredixcan_file,
            tissue_proj,
            OUTPUT_PREDICTIONS_DIR,
            PREDICTION_METHOD,
            doids_in_gold_standard,
            ukb_efo,
            efo_xrefs,
            do_xrefs,
            n_top_conditions=ntc,
            use_abs=True,
        )

    print('')

spredixcan-mashr-zscores-Adipose_Subcutaneous-projection-gtex.pkl
  shape: (578, 4091)
  predicting all_genes...
    shape: (1170, 4091)


    saved to: /home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/08_prediction_module_based_gtex/lincs/predictions/dotprod_neg/spredixcan-mashr-zscores-Adipose_Subcutaneous-projection-gtex-all_genes-prediction_scores.h5
  predicting top_5_genes...


    shape: (1170, 4091)


    saved to: /home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/08_prediction_module_based_gtex/lincs/predictions/dotprod_neg/spredixcan-mashr-zscores-Adipose_Subcutaneous-projection-gtex-top_5_genes-prediction_scores.h5
  predicting top_10_genes...


    shape: (1170, 4091)


...

    saved to: /home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/08_prediction_module_based_gtex/lincs/predictions/dotprod_neg/spredixcan-mashr-zscores-Whole_Blood-projection-gtex-top_10_genes-prediction_scores.h5
  predicting top_25_genes...


    shape: (1170, 4091)


    saved to: /home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/08_prediction_module_based_gtex/lincs/predictions/dotprod_neg/spredixcan-mashr-zscores-Whole_Blood-projection-gtex-top_25_genes-prediction_scores.h5
  predicting top_50_genes...


    shape: (1170, 4091)


    saved to: /home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/08_prediction_module_based_gtex/lincs/predictions/dotprod_neg/spredixcan-mashr-zscores-Whole_Blood-projection-gtex-top_50_genes-prediction_scores.h5

